# 含持有期零段反转的八列表与连续诊断

本 Notebook 先调用当前最终冻结包生成原始八列表，再只替换 `0转-1` 和 `0转+1` 两列。

- 零段信号在形成日收盘计算，下一实际交易日执行。
- 执行日的零段信号使用当前冻结参数中的完整持仓机路径，信号发出后连续输出 1。下侧使用 H03，上侧使用 H04；最短持有、最长持有、释放阈值和冷却期均沿用冻结参数。
- 如果持有路径对应的基础三状态不是 0，则把该日零段信号置为 0。
- `0转-1` 与 `0转+1` 同时为 1 的情况暂不处理，保留两列各自的冻结持有结果。
- `三状态`、`+1反转`、`-1反转`、`大涨`、`大跌` 不做修改。
- 在不改变八列表的前提下，额外输出 `连续诊断输出.csv`：包含 `state_strength`、`state_phase`、快慢引擎，以及 V38/V57、V156/V189 的连续分数和冻结阈值距离。

运行前请确保环境使用冻结包要求的 `pandas>=2.0,<3.0`，并设置 `COMPANY_SPOT_PATH`。

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get(
    'COMPANY_SPOT_PATH',
    '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet',
).strip()
if not SPOT_TEXT:
    raise RuntimeError(
        '请先设置 COMPANY_SPOT_PATH 为唯一的原始现货 Parquet/CSV/TSV 文件路径。'
    )

from generate_compact_output import generate_compact_output, resolve_spot

SPOT_PATH = resolve_spot(SPOT_TEXT)
RUN_ROOT = Path(
    os.environ.get(
        'HOLDING_OUTPUT_DIR',
        str(PACKAGE_ROOT / 'runtime_outputs_holding_period'),
    )
).expanduser().resolve()
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(f'冻结包目录：{PACKAGE_ROOT}')
print(f'现货输入：{SPOT_PATH}')
print(f'输出目录：{RUN_ROOT}')

冻结包目录：/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包
现货输入：/Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824_1750/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet
输出目录：/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包/runtime_outputs_holding_period


## 1. 生成当前冻结包的原始八列表

这一格调用生产入口，生成未加入持有期的原始八列表。它不会重选候选参数。

In [2]:
record = generate_compact_output(SPOT_PATH, RUN_ROOT)
EVENT_EIGHT_PATH = RUN_ROOT / '最终执行日简表.csv'
event_eight = pd.read_csv(EVENT_EIGHT_PATH, encoding='utf-8-sig')
event_eight['实际执行日'] = pd.to_datetime(event_eight['实际执行日'], errors='raise').dt.normalize()

COMPACT_COLUMNS = ['实际执行日', '三状态', '+1反转', '-1反转', '0转-1', '0转+1', '大涨', '大跌']
if list(event_eight.columns) != COMPACT_COLUMNS:
    raise AssertionError(f'原始八列表列顺序不符：{list(event_eight.columns)}')

print(f'原始八列表已生成：{EVENT_EIGHT_PATH}')
print(f'行数：{len(event_eight):,}')

[非零冻结] 启动
[非零退出] 读取冻结 V55；只计算最终 score_02，不重建候选池
[非零退出] V55 冻结信号完成；事件数=45
[非零退出] 读取冻结 V80；只计算最终 score_02，不重建候选池
[非零退出] V80 冻结信号完成；事件数=62
[非零退出] output=/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包/runtime_outputs_holding_period/_engine_outputs/remote_nonzero_five_columns.csv
[非零退出] rows=2097 date=2018-01-03 -> 2026-08-25
[零段反转冻结] 启动
[零段反转] 读取唯一远端现货并从冻结八状态公式生成基础状态
[零段反转-down] 运行冻结 V38_down_s01_a1_q0.85_c1_H03（不扫描候选、不读取未来标签）
[零段反转-up] 运行冻结 V57_up_s01_a1_q0.85_c1_H04（不扫描候选、不读取未来标签）
[零段反转] output=/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包/runtime_outputs_holding_period/_engine_outputs/remote_zero_transfer_predictions.csv; rows=2097; execution=2018-01-03 -> 2026-08-25
[大跌冻结] 启动
[O2O down 11:19:05] 1/4 读取唯一现货并构造因果特征
[O2O down 11:19:08] 1/4 完成：rows=4766，Development=1213，Validation=482
[O2O down 11:19:08] 2/4 只计算已冻结参数：V156:base_0621_cov_0.075；不重建候选池
[O2O down 11:19:08] 3/4 用固定阈值生成冻结预测，之后才读取 Test
[O2O down 11:19:08] 4/4 冻结参数逐日结果已生成
[大涨大跌-down] prediction_output=/Users/hzy/Deskto

## 2. 重建冻结零段信号的连续持有路径

这里直接调用冻结运行包中的评分和持仓机函数，不读取未来收益标签。`holding_path` 是冻结参数实际产生的连续持有路径。

In [3]:
from run_zero_transfer_frozen import (
    FROZEN_ZERO_TRANSFER,
    _effective_dates,
    _frozen_signal_path,
)
from spot_panel import load_spot_panel
from zero_transfer.logic_features import compute_logic_scores

spot, panel, spot_audit = load_spot_panel(SPOT_PATH)
effective_dates, pending_effective = _effective_dates(panel)

entry_paths = {}
holding_paths = {}
freeze_rows = {}
for side in ('down', 'up'):
    freeze = FROZEN_ZERO_TRANSFER[side]
    scores = compute_logic_scores(
        str(freeze['source_version']),
        spot,
        panel,
        int(freeze['direction']),
    )
    score_column = str(freeze['score_column'])
    if score_column not in scores.columns:
        raise KeyError(f'{side} 冻结评分列不存在：{score_column}')
    selected, holding = _frozen_signal_path(
        panel,
        scores[score_column].to_numpy(dtype=float),
        freeze,
    )
    entry_paths[side] = selected.astype(bool)
    holding_paths[side] = holding.astype(bool)
    freeze_rows[side] = freeze

base_state = pd.to_numeric(panel['state'], errors='raise').astype(int).to_numpy()
base_is_zero = base_state == 0

# 先保留冻结持有路径，再按执行日对应的基础状态做冲突置零。
down_conflict = holding_paths['down'] & ~base_is_zero
up_conflict = holding_paths['up'] & ~base_is_zero
down_holding = holding_paths['down'] & base_is_zero
up_holding = holding_paths['up'] & base_is_zero

holding_detail = pd.DataFrame({
    '形成日': pd.to_datetime(panel['formation_date'], errors='raise').dt.normalize(),
    '实际执行日': pd.to_datetime(effective_dates),
    '基础三状态': base_state,
    '下侧事件信号': entry_paths['down'].astype('int8'),
    '下侧冻结持有路径': holding_paths['down'].astype('int8'),
    '0转-1最终信号': down_holding.astype('int8'),
    '上侧事件信号': entry_paths['up'].astype('int8'),
    '上侧冻结持有路径': holding_paths['up'].astype('int8'),
    '0转+1最终信号': up_holding.astype('int8'),
})

if holding_detail['实际执行日'].duplicated().any():
    raise AssertionError('持有期明细的实际执行日重复')

display(pd.DataFrame([
    {
        '方向': side,
        '候选编号': str(freeze_rows[side]['candidate_id']),
        '持有包': str(freeze_rows[side]['holding_package']['package_id']),
        '最短持有日': int(freeze_rows[side]['holding_package']['min_hold_days']),
        '最长持有日': int(freeze_rows[side]['holding_package']['max_hold_days']),
        '事件信号数': int(entry_paths[side].sum()),
        '冻结持有日数': int(holding_paths[side].sum()),
        '基础状态冲突后置零日数': int((down_conflict if side == 'down' else up_conflict).sum()),
    }
    for side in ('down', 'up')
]))

,方向,候选编号,持有包,最短持有日,最长持有日,事件信号数,冻结持有日数,基础状态冲突后置零日数
0,down,V38_down_s01_a1_q0.85_c1_H03,H03,3,10,137,333,9
1,up,V57_up_s01_a1_q0.85_c1_H04,H04,5,20,86,420,30


## 3. 合并为含持有期的八列表

只替换两个零段列；两个零段方向同日同时为 1 时不做额外冲突处理。

In [4]:
detail_by_execution = holding_detail.set_index('实际执行日')
event_dates = pd.DatetimeIndex(event_eight['实际执行日'])
if set(event_dates) != set(detail_by_execution.index):
    raise AssertionError('原始八列表与持有期明细的执行日集合不一致')

# 官方事件信号应当与同一冻结路径的 entry_path 一致；先做日期对齐审计。
entry_check = detail_by_execution.reindex(event_dates)
if not np.array_equal(event_eight['三状态'].to_numpy(dtype=int), entry_check['基础三状态'].to_numpy(dtype=int)):
    raise AssertionError('基础三状态的形成日值与八列表执行日映射不一致')
if not np.array_equal(event_eight['0转-1'].to_numpy(dtype=int), entry_check['下侧事件信号'].to_numpy(dtype=int)):
    raise AssertionError('0转-1 事件信号与冻结运行结果不一致')
if not np.array_equal(event_eight['0转+1'].to_numpy(dtype=int), entry_check['上侧事件信号'].to_numpy(dtype=int)):
    raise AssertionError('0转+1 事件信号与冻结运行结果不一致')

holding_eight = event_eight.copy()
holding_eight['0转-1'] = entry_check['0转-1最终信号'].to_numpy(dtype='int8')
holding_eight['0转+1'] = entry_check['0转+1最终信号'].to_numpy(dtype='int8')

for column in ('三状态', '+1反转', '-1反转', '大涨', '大跌'):
    if not np.array_equal(holding_eight[column].to_numpy(), event_eight[column].to_numpy()):
        raise AssertionError(f'{column} 不应被持有期口径修改')

OUTPUT_PATH = RUN_ROOT / '含持有期八列表.csv'
DETAIL_PATH = RUN_ROOT / '零段反转持有期明细.csv'
METADATA_PATH = RUN_ROOT / '含持有期八列表_运行记录.json'

holding_eight.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
holding_detail.to_csv(DETAIL_PATH, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')

both_entry_or_holding = (holding_eight['0转-1'].eq(1) & holding_eight['0转+1'].eq(1))
metadata = {
    'input_spot': str(SPOT_PATH),
    'source_event_eight': str(EVENT_EIGHT_PATH),
    'output_file': str(OUTPUT_PATH),
    'holding_detail_file': str(DETAIL_PATH),
    'columns': COMPACT_COLUMNS,
    'date_rule': 'formation_date t close -> actual execution date t+1; hold path starts on the execution date',
    'zero_transfer_policy': 'use frozen holding_path, then set the zero-transfer value to 0 when the aligned base three-state is not 0',
    'same_day_down_up_policy': 'not handled; retain both independently generated columns',
    'same_day_both_one_days': int(both_entry_or_holding.sum()),
    'pending_latest_execution_display': bool(pending_effective),
    'freeze': {
        side: {
            'candidate_id': str(freeze_rows[side]['candidate_id']),
            'source_version': str(freeze_rows[side]['source_version']),
            'direction': int(freeze_rows[side]['direction']),
            'holding_package': dict(freeze_rows[side]['holding_package']),
            'threshold_from_development': float(freeze_rows[side]['threshold_from_development']),
            'release_threshold': float(freeze_rows[side]['release_threshold']),
        }
        for side in ('down', 'up')
    },
    'counts': {
        'rows': int(len(holding_eight)),
        'down_event_days': int(entry_paths['down'].sum()),
        'up_event_days': int(entry_paths['up'].sum()),
        'down_raw_holding_days': int(holding_paths['down'].sum()),
        'up_raw_holding_days': int(holding_paths['up'].sum()),
        'down_conflict_zeroed_days': int(down_conflict.sum()),
        'up_conflict_zeroed_days': int(up_conflict.sum()),
        'down_final_one_days': int(holding_eight['0转-1'].sum()),
        'up_final_one_days': int(holding_eight['0转+1'].sum()),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print(f'已输出：{OUTPUT_PATH}')
print(f'持有期明细：{DETAIL_PATH}')
print(f'运行记录：{METADATA_PATH}')

已输出：/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包/runtime_outputs_holding_period/含持有期八列表.csv
持有期明细：/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包/runtime_outputs_holding_period/零段反转持有期明细.csv
运行记录：/Users/hzy/Desktop/0817合并查看/20260825_1111_冻结中证500输出包/runtime_outputs_holding_period/含持有期八列表_运行记录.json


## 4. 最终检查

In [5]:
if list(holding_eight.columns) != COMPACT_COLUMNS:
    raise AssertionError('最终八列表列顺序不正确')
if holding_eight['实际执行日'].duplicated().any():
    raise AssertionError('最终八列表执行日重复')
if not holding_eight['实际执行日'].is_monotonic_increasing:
    raise AssertionError('最终八列表执行日未排序')
if not holding_eight['三状态'].isin([-1, 0, 1]).all():
    raise AssertionError('三状态出现非法值')
if not holding_eight[['+1反转', '-1反转', '0转-1', '0转+1', '大涨', '大跌']].isin([0, 1]).all().all():
    raise AssertionError('八列表信号出现非 0/1 值')
if ((holding_eight['三状态'].ne(0)) & holding_eight['0转-1'].eq(1)).any():
    raise AssertionError('0转-1 仍存在基础状态冲突')
if ((holding_eight['三状态'].ne(0)) & holding_eight['0转+1'].eq(1)).any():
    raise AssertionError('0转+1 仍存在基础状态冲突')

print('全部检查通过。')
print('注意：0转-1 与 0转+1 同时为 1 的情况按要求暂未处理。')
display(pd.DataFrame({
    '列': ['0转-1', '0转+1'],
    '事件日 1 数量': [int(event_eight['0转-1'].sum()), int(event_eight['0转+1'].sum())],
    '持有期最终 1 数量': [int(holding_eight['0转-1'].sum()), int(holding_eight['0转+1'].sum())],
    '基础状态冲突置零': [int(down_conflict.sum()), int(up_conflict.sum())],
}))

全部检查通过。
注意：0转-1 与 0转+1 同时为 1 的情况按要求暂未处理。


,列,事件日 1 数量,持有期最终 1 数量,基础状态冲突置零
0,0转-1,137,324,9
1,0转+1,86,390,30


## 日常最终输出（放在 Notebook 最底部）

下面先显示干净的含持有期精简八列表，再显示同一形成日对应的全部有用连续值。连续值表的每一行仍然是形成日收盘计算、下一实际交易日执行；最新一行的数值来自最新形成日真实计算，不用占位值。阈值说明紧跟在连续值表下方。

In [6]:
# 只在最后集中展示两个日常要看的表；中间计算单元格不再插入长表格。
final_eight = pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig')
final_continuous = pd.read_csv(RUN_ROOT / '连续诊断输出.csv', encoding='utf-8-sig')
print('一、干净的含持有期精简八列表（文件保存全部历史；这里显示最新20行）')
display(final_eight.tail(20))
print('二、连续诊断输出（文件保存全部历史；这里显示最新20行）')
display(final_continuous.tail(20).round(6))
print('三、连续值和冻结阈值说明')
display(pd.DataFrame([
    ['state_strength', '冻结三状态规则证据强度；-1/0/1分别表示下行、中性、上行规则支持，范围[-1,1]，不是收益率或概率', '无单独触发阈值'],
    ['state_phase', '状态机阶段：entry=进入，continuation=持续，pending_switch=等待冻结确认', '字符串标签'],
    ['fast_engine / slow_engine', '三状态内部快/慢方向引擎连续值，形成日收盘计算', '[0,1]；不能单独当作预测概率'],
    ['-1转0_score', 'V55 下行状态退出连续分数；距阈值=score-冻结阈值', '冻结阈值 0.815842643776842；确认日数 2'],
    ['+1转0_score', 'V80 上行状态退出连续分数；距阈值=score-冻结阈值', '冻结阈值 0.4920211306764882；确认日数 1'],
    ['0转-1_score', 'V38 零段下行连续分数；距阈值=score-冻结阈值', '入口阈值 1.2294466376933055；释放阈值 0.6645986031220402'],
    ['0转+1_score', 'V57 零段上行连续分数；距阈值=score-冻结阈值', '入口阈值 4.671980676328502；释放阈值 4.606280193236715'],
    ['大跌_score', 'V156 大跌预警连续分数；距阈值=score-冻结阈值', '冻结阈值 0.6832148298881413；大于等于阈值才是大跌=1'],
    ['大涨_score', 'V189 大涨预测连续分数；距阈值=score-冻结阈值', '冻结阈值 0.8135114753699175；大于等于阈值才是大涨=1'],
], columns=['字段', '含义', '阈值/口径']))

一、干净的含持有期精简八列表（文件保存全部历史；这里显示最新20行）


,实际执行日,三状态,+1反转,-1反转,0转-1,0转+1,大涨,大跌
2077,2026-07-29,-1,0,0,0,0,0,0
2078,2026-07-30,-1,0,0,0,0,0,0
2079,2026-07-31,-1,0,0,0,0,0,0
2080,2026-08-03,-1,0,1,0,0,0,0
2081,2026-08-04,0,0,0,0,0,0,0
2082,2026-08-05,0,0,0,0,0,0,0
2083,2026-08-06,0,0,0,0,0,0,0
2084,2026-08-07,0,0,0,0,0,0,0
2085,2026-08-10,0,0,0,0,0,0,0
2086,2026-08-11,0,0,0,1,0,0,0


二、连续诊断输出（文件保存全部历史；这里显示最新20行）


,形成日,实际执行日,三状态,state_strength,state_phase,fast_engine,slow_engine,direction_score,direction_score_continuous,direction_score_band,...,0转+1_距释放阈值,0转+1_事件信号,大涨_score,大涨_冻结阈值,大涨_距阈值,大涨_预测,大跌_score,大跌_冻结阈值,大跌_距阈值,大跌_预测
2077,2026-07-28,2026-07-29,-1,-0.500000,continuation,0.059524,0.062500,0.039306,0.056151,0.000000,...,-9.217391,0,0.359983,0.813511,-0.453529,0,0.598413,0.683215,-0.084802,0
2078,2026-07-29,2026-07-30,-1,-0.000000,pending_switch,0.097222,0.249008,0.128056,0.182937,0.000000,...,0.014493,0,0.321141,0.813511,-0.492371,0,0.616886,0.683215,-0.066329,0
2079,2026-07-30,2026-07-31,-1,-0.422049,continuation,0.060516,0.135913,0.070486,0.100694,0.000000,...,0.159420,0,0.512089,0.813511,-0.301422,0,0.567055,0.683215,-0.116160,0
2080,2026-07-31,2026-08-03,-1,-0.000000,pending_switch,0.266865,0.087302,0.108819,0.139385,0.037500,...,0.077295,0,0.559531,0.813511,-0.253981,0,0.426030,0.683215,-0.257185,0
2081,2026-08-03,2026-08-04,0,0.039683,entry,0.272817,0.098214,0.113681,0.146329,0.037500,...,-9.231884,0,0.355160,0.813511,-0.458351,0,0.405122,0.683215,-0.278093,0
2082,2026-08-04,2026-08-05,0,0.055272,entry,0.365079,0.112103,0.156528,0.191468,0.075000,...,0.009662,0,0.308990,0.813511,-0.504522,0,0.398740,0.683215,-0.284475,0
2083,2026-08-05,2026-08-06,0,0.139361,continuation,0.346230,0.499008,0.382153,0.428075,0.275000,...,0.062802,0,0.298265,0.813511,-0.515247,0,0.419392,0.683215,-0.263823,0
2084,2026-08-06,2026-08-07,0,0.112200,continuation,0.325397,0.502976,0.354167,0.430952,0.175000,...,0.014493,0,0.332461,0.813511,-0.481051,0,0.450032,0.683215,-0.233183,0
2085,2026-08-07,2026-08-10,0,0.113379,continuation,0.374008,0.515873,0.392708,0.469940,0.212500,...,-9.260870,0,0.259943,0.813511,-0.553569,0,0.450932,0.683215,-0.232282,0
2086,2026-08-10,2026-08-11,0,0.292895,continuation,0.454365,0.485119,0.419583,0.477976,0.283333,...,-0.043478,0,0.268610,0.813511,-0.544901,0,0.445659,0.683215,-0.237556,0


三、连续值和冻结阈值说明


,字段,含义,阈值/口径
0,state_strength,"冻结三状态规则证据强度；-1/0/1分别表示下行、中性、上行规则支持，范围[-1,1]，不是...",无单独触发阈值
1,state_phase,状态机阶段：entry=进入，continuation=持续，pending_switch=...,字符串标签
2,fast_engine / slow_engine,三状态内部快/慢方向引擎连续值，形成日收盘计算,"[0,1]；不能单独当作预测概率"
3,-1转0_score,V55 下行状态退出连续分数；距阈值=score-冻结阈值,冻结阈值 0.815842643776842；确认日数 2
4,+1转0_score,V80 上行状态退出连续分数；距阈值=score-冻结阈值,冻结阈值 0.4920211306764882；确认日数 1
5,0转-1_score,V38 零段下行连续分数；距阈值=score-冻结阈值,入口阈值 1.2294466376933055；释放阈值 0.6645986031220402
6,0转+1_score,V57 零段上行连续分数；距阈值=score-冻结阈值,入口阈值 4.671980676328502；释放阈值 4.606280193236715
7,大跌_score,V156 大跌预警连续分数；距阈值=score-冻结阈值,冻结阈值 0.6832148298881413；大于等于阈值才是大跌=1
8,大涨_score,V189 大涨预测连续分数；距阈值=score-冻结阈值,冻结阈值 0.8135114753699175；大于等于阈值才是大涨=1
